# 05: Embedding Paradigms

Builds and compares the three Section 3.4 embedding paradigms (FT-Transformer, SubTab, SCARF) on top of the already-fitted encoders from `04_feature_encoding.ipynb`.

## Setup

In [ ]:
%matplotlib inline

import os

# torch bundles its own OpenMP runtime (libiomp5md.dll) while
# numpy/pandas/scikit-learn pull in MKL's (libomp.dll/libiomp5md.dll) --
# loading both into one process aborts the kernel outright ("OMP: Error
# #15: Initializing ..., but found ... already initialized"), not a
# Python exception, so it shows up as a bare kernel crash with no
# traceback. Must be set before numpy/torch are imported anywhere below.
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
from pathlib import Path

from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import config

FINAL_DIR = config.FINAL_DIR

sns.set_theme(style="whitegrid")

from src.features.target_split import build_feature_target_split
from src.features.pipeline import FittedEncoders

## 1. Load active candidate

In [ ]:
CANDIDATES = {
    "br_gt0": {
        "train": FINAL_DIR / "tmdb_br_gt0_train.parquet",
        "val": FINAL_DIR / "tmdb_br_gt0_val.parquet",
        "test": FINAL_DIR / "tmdb_br_gt0_test.parquet",
    },
    "nbr_gt5": {
        "train": FINAL_DIR / "tmdb_nbr_gt5_train.parquet",
        "val": FINAL_DIR / "tmdb_nbr_gt5_val.parquet",
        "test": FINAL_DIR / "tmdb_nbr_gt5_test.parquet",
    },
}

In [ ]:
# Single switch: "br_gt0" is the primary candidate, "nbr_gt5" the
# fallback. Every later cell reads train_df/val_df/test_df (or
# ACTIVE_CANDIDATE for naming saved artifacts) instead of a literal
# candidate name, so switching candidates means editing only this line.
ACTIVE_CANDIDATE = "br_gt0"
# ACTIVE_CANDIDATE = "nbr_gt5"

In [ ]:
train_df = pd.read_parquet(CANDIDATES[ACTIVE_CANDIDATE]["train"])
val_df = pd.read_parquet(CANDIDATES[ACTIVE_CANDIDATE]["val"])
test_df = pd.read_parquet(CANDIDATES[ACTIVE_CANDIDATE]["test"])

FINAL_SPLITS = {"train": train_df, "val": val_df, "test": test_df}

train_split = build_feature_target_split(train_df)

ARTIFACTS_DIR = REPO_ROOT / "data" / "artifacts" / ACTIVE_CANDIDATE
fitted_encoders = FittedEncoders.fit(FINAL_SPLITS, train_split, ARTIFACTS_DIR)

print(f"Active candidate: {ACTIVE_CANDIDATE}")
print(f"  train: {train_df.shape}")
print(f"  val:   {val_df.shape}")
print(f"  test:  {test_df.shape}")
print(f"fitted_encoders.dims: {fitted_encoders.dims}")

## 2. Paradigm hyperparameter configs

Instantiate all four `src/models/paradigm_config.py` dataclasses and print every field before anything trains on them -- a quick eyeball check against the papers they're cited from.

In [ ]:
import dataclasses

from src.models.paradigm_config import (
    FTTransformerConfig,
    SubTabConfig,
    SCARFConfig,
    SharedTrainingConfig,
)

ft_transformer_config = FTTransformerConfig()
subtab_config = SubTabConfig()
scarf_config = SCARFConfig()
shared_training_config = SharedTrainingConfig()

for name, cfg in [
    ("FTTransformerConfig", ft_transformer_config),
    ("SubTabConfig", subtab_config),
    ("SCARFConfig", scarf_config),
    ("SharedTrainingConfig", shared_training_config),
]:
    print(f"--- {name} ---")
    for key, value in dataclasses.asdict(cfg).items():
        print(f"  {key}: {value!r}")
    print()

## 3. FT-Transformer

How do the frozen SBERT vectors enter the token sequence?

`overview`/`original_title` each arrive from `fitted_encoders.transform()` as a frozen 768-d SBERT vector (`fitted_encoders.dims["overview"] == fitted_encoders.dims["original_title"] == 768`). Every other token in FT-Transformer's sequence -- numeric (PLE-then-linear), categorical (embedding lookup), list-pooled -- is built at `d_token`. Two options for how the text tokens join that sequence:

**(a) Used as-is, as two large 768-d tokens.**
- Pros: no information loss from the frozen SBERT vector -- the full representation reaches self-attention; no extra trainable compression step between the frozen encoder and the model.
- Cons: self-attention requires every token in the sequence to share the same width, so this is only actually implementable by setting `d_token=768` *globally* -- not just for the two text tokens, but for numeric, categorical, and list tokens too. That silently abandons Gorishniy et al.'s own `d_token=192` defaulf. It also inflates every attention block's parameter count and compute cost by roughly 768/192 = 4x per token width dimension, for a width inherited from SBERT's arbitrary output size, not chosen for FT-Transformer.

**(b) Projected down to `d_token` (192) via a small trainable `nn.Linear(768, d_token)` per text field.**
- Pros: keeps `d_token=192`, respecting Table 12's default; every token in the sequence stays at one uniform width, matching FT-Transformer's architecture as published rather than special-casing it; the projection is itself trainable, so the model learns which directions in the frozen 768-d SBERT space matter for this task instead of passing the whole undifferentiated vector through; parameter/compute cost stays at the paper's intended scale.
- Cons: introduces a small lossy compression (768 -> 192) before self-attention ever sees the text tokens; adds new trainable parameters (768x192 + bias, per text field) that have to be learned from this dataset, unlike SBERT's own frozen weights; whether the two text fields share one projection or each gets its own is a further decision left for the actual model code, not resolved here.

**Decision: option (b).** Self-attention's shared-width requirement means option (a) was never really a free choice -- it was a choice to override `d_token` globally. `nn.Linear(768, d_token)` projections for `overview`/`original_title` proceed as the design; no model code is written yet, this cell records the decision and reasoning only.

### Sanity check: n_tokens

`fitted_encoders.dims` has one entry per `transform()` key (9 for `br_gt0`: `numeric`, `original_language`, `overview`, `original_title`, and 5 list fields) -- but `n_tokens` is NOT `len(dims)`, because the single `"numeric"` key represents one *combined* width that `FeatureTokenizer` re-slices back into one token per `numeric_cols` entry plus one token per `bypass_cols` entry. Every other `dims` key maps to exactly one token. So:

`n_tokens = len(numeric_cols) + len(bypass_cols) + 1 (language) + len(list_cols) + 2 (text)`

equivalently `(len(dims) - 1) + len(numeric_cols) + len(bypass_cols)` -- computed both ways below and checked against `FeatureTokenizer`'s actual output.

In [ ]:
from src.models.ft_transformer import FeatureTokenizer

tokenizer = FeatureTokenizer(fitted_encoders, d_token=ft_transformer_config.d_token)
batch = fitted_encoders.transform(train_df)
tokens = tokenizer(batch)

column_groups = fitted_encoders.column_groups
n_tokens_by_groups = (
    len(column_groups.numeric_cols)
    + len(column_groups.bypass_cols)
    + 1  # original_language
    + len(column_groups.list_cols)
    + 2  # overview, original_title
)
n_tokens_by_dims = (
    (len(fitted_encoders.dims) - 1)  # every dims key except "numeric" is one token
    + len(column_groups.numeric_cols)
    + len(column_groups.bypass_cols)
)

print(f"dims keys ({len(fitted_encoders.dims)}): {list(fitted_encoders.dims.keys())}")
print(f"n_tokens via column_groups: {n_tokens_by_groups}")
print(f"n_tokens via dims key count: {n_tokens_by_dims}")
print(f"FeatureTokenizer actual output: {tokens.shape[1]}")

assert n_tokens_by_groups == n_tokens_by_dims == tokens.shape[1], (
    f"n_tokens mismatch: by_groups={n_tokens_by_groups}, "
    f"by_dims={n_tokens_by_dims}, actual={tokens.shape[1]}"
)
print("PASSED: all three counts agree")

Beyond `n_tokens` alone, `FeatureTokenizer.forward()`'s full output must be exactly `(batch_size, n_tokens, d_token)` -- one token per feature, each `d_token` wide, for every row in the batch (Prompt 2.2's check). Reusing `tokenizer`/`batch`/`tokens` from the cell above, `batch_size` here is `len(train_df)` since `batch = fitted_encoders.transform(train_df)` ran on the whole training set as one call, not a mini-batch.

In [ ]:
n_rows = len(train_df)
expected_shape = (n_rows, n_tokens_by_groups, ft_transformer_config.d_token)

print(f"batch size (rows in train_df): {n_rows}")
print(f"FeatureTokenizer output shape: {tuple(tokens.shape)}")
print(f"expected shape: {expected_shape}")

assert tokens.shape == expected_shape, (
    f"shape mismatch: got {tuple(tokens.shape)}, expected {expected_shape}"
)
print("PASSED: output shape is (batch_size, n_tokens, d_token) as predicted")

### Sanity check: parameter count

Instantiate `FTTransformerModel` from `fitted_encoders.dims` (drives `FeatureTokenizer`'s per-field embedding/projection tables) and `ft_transformer_config`, then count total parameters. Gorishniy et al. (2021, Table 12) report 929K parameters for FT-Transformer -- but that's on their own benchmark datasets, with their own feature counts and vocabulary sizes, not this one. This dataset has more embedding-heavy fields (5 list fields plus `original_language`, each its own vocabulary -sized `nn.Embedding` table) and two 768->192 text projections, so matching 929K isn't the bar -- staying in the same order of magnitude is.

In [ ]:
from src.models.ft_transformer import FTTransformerModel

model = FTTransformerModel(fitted_encoders, ft_transformer_config)
total_params = sum(p.numel() for p in model.parameters())

print(f"total parameters: {total_params:,}")
print(f"Gorishniy et al. (2021), Table 12: 929,000")
print(f"ratio: {total_params / 929_000:.2f}x")

### Train FT-Transformer

In [ ]:
from src.models.train_ft_transformer import train_ft_transformer

model, train_losses, val_losses = train_ft_transformer(
    model, train_df, val_df, fitted_encoders, ft_transformer_config, shared_training_config
)

print(
    f"Epochs run: {len(train_losses)} "
    f"(max_epochs={shared_training_config.max_epochs}, patience={shared_training_config.patience})"
)
print(f"Final train MSE: {train_losses[-1]:.4f}")
print(f"Val MSE (restored/best epoch): {min(val_losses):.4f}")

**Ad hoc sanity check only -- not the final asset.** Prompt 5.5's shared diagnostics logging will later replace this plot with the identical-across-paradigms version (same axes/styling for FT-Transformer, SubTab, and SCARF alike). This cell just gives a quick look at whether `train_losses`/`val_losses` moved the way early stopping's logic assumes, before that shared logger exists.

In [ ]:
fig, ax = plt.subplots()
ax.plot(train_losses, label="train MSE")
ax.plot(val_losses, label="val MSE")
ax.set_xlabel("epoch")
ax.set_ylabel("MSE (vote_average)")
ax.set_title("FT-Transformer training curve (ad hoc -- see note above)")
ax.legend()
plt.show()

### Save FT-Transformer embeddings

`model.embed()` (Eq. 3.9) run over `train_df`/`val_df`/`test_df` in batches, under `torch.no_grad()` -- inference only, no gradient tracking needed once the model is trained. Each split's `(N, d_token)` embedding matrix is saved to `ARTIFACTS_DIR / "models" / "ft_transformer" / f"{split_name}.parquet"`, mirroring `04_feature_encoding.ipynb`'s `RAW_MODEL_DIR`/`CLASSICAL_MODEL_DIR` pattern: one row per input row, in the same order as the split's DataFrame -- no `id`/target column saved alongside, same as the raw/classical baselines, so downstream code aligns by row position against the original split parquet files. Columns are named `dim_0..dim_{d_token-1}`; there's no natural per-dimension feature name the way raw/classical's one-hot/multi-hot columns have one.

In [ ]:
import torch

FT_TRANSFORMER_MODEL_DIR = ARTIFACTS_DIR / "models" / "ft_transformer"
FT_TRANSFORMER_MODEL_DIR.mkdir(parents=True, exist_ok=True)

model.eval()
for split_name, df in FINAL_SPLITS.items():
    batch = fitted_encoders.transform(df)
    embedding_chunks = []
    with torch.no_grad():
        for start in range(0, len(df), ft_transformer_config.batch_size):
            end = min(start + ft_transformer_config.batch_size, len(df))
            chunk = {key: value[start:end] for key, value in batch.items()}
            embedding_chunks.append(model.embed(chunk).numpy())
    embeddings = np.concatenate(embedding_chunks, axis=0)

    embeddings_df = pd.DataFrame(
        embeddings, columns=[f"dim_{i}" for i in range(embeddings.shape[1])]
    )
    path = FT_TRANSFORMER_MODEL_DIR / f"{split_name}.parquet"
    embeddings_df.to_parquet(path, index=False)
    print(f"Saved: {path} ({embeddings_df.shape})")